In [37]:
import re


# collect short definitions for vocabulary
lexicalentries = {}
verbarguments = {}
lexfile = "/Users/gcrane/github/Thucydides-new-working-materials/GreekMorphApril18.csv"
with open(lexfile) as f:
    for line in f:

        #print(line)
        #line = line.strip('"')
        fields = line.split(",")
        fields[1] = fields[1].strip('"')
        fields[2] = fields[2].strip('\n')
        fields[2] = fields[2].strip('"')
        lexicalentries[fields[1]] = fields[2]

f.close()

lemmasbybook = {}
lemmasbywork = {}
lemmastot = {}

tokensbybook = {}
tokensbywork = {}
tokenstot = 0

lemmasbywork['1'] = {}
lemmasbywork['2'] = {}

tokensbywork['1'] = 0
tokensbywork['2'] = 0
for i in range(1,25):
    tokensbybook["1:"+str(i)] = 0
    tokensbybook["2:"+str(i)] = 0
    lemmasbybook["1:"+str(i)] = {}
    lemmasbybook["2:"+str(i)] = {}

def addentry(curitem,curdict):
    if(curitem in curdict):
        curdict[curitem] = curdict[curitem] + 1
    else:
        curdict[curitem] = 1

def getlemmasbybook(curwork,fname):
    global lemmasbybook
    global lemmasbywork
    global lemmastot
    
    global tokensbybook
    global tokensbywork
    global tokenstot
    f = open(fname)
    
    for l in f:
        m = re.search('Ref=([0-9]+)\.([0-9]+)',l)
        if(not m):
            continue
        if(re.search('PUNCT',l)):
            continue
        curbook = curwork + ':' + str(m[1])
        curline = str(m[2])
        args = l.split()
        addentry(args[2],lemmasbybook[curbook])
        addentry(args[2],lemmasbywork[curwork])
        addentry(args[2],lemmastot)
        
        tokensbybook[curbook] = tokensbybook[curbook] + 1
        tokensbywork[curwork] = tokensbywork[curwork] + 1
        tokenstot = tokenstot + 1
        
        #print(args[2])
    
    
    f.close()
    
    
getlemmasbybook('1','tlg001/tlg0012.tlg001.daphne_tb-grc1.1-6.conllu')
getlemmasbybook('2','tlg002/tlg0012.tlg002.daphne_tb-grc1.7-12.conllu')

In [40]:
outf = open('il6od8voc.tsv','w')
for foo in sorted(lemmastot,key=lemmastot.get,reverse=True):
    
    if(foo in lemmasbybook['1:6']):
        il6 = lemmasbybook['1:6'][foo]
        il6n = (1000*il6/tokensbybook['1:6'])
        il6f = re.sub('(\.[0-9][0-9])[0-9]*','\g<1>',str(il6n))
    else:
        il6 = 0
        il6f = '0'
    if(foo in lemmasbybook['2:8']):
        od8 = lemmasbybook['2:8'][foo]
        od8n = (1000*od8/tokensbybook['2:8'])
        od8f = re.sub('(\.[0-9][0-9])[0-9]*','\g<1>',str(od8n))
    else:
        od8 = 0
        
    tmp = 1000*lemmastot[foo]/tokenstot
    totf = re.sub('(\.[0-9][0-9])[0-9]*','\g<1>',str(tmp))
    
    if(foo in lexicalentries):
        curgloss = lexicalentries[foo]
    else:
        curgloss = 'NoDef'
    if(od8 or il6):
        print(foo,curgloss, lemmastot[foo],totf,il6,il6f,od8,od8f,sep='\t',file=outf)
outf.close()    